# 🧬 Mechanogenomic Virtual Cell — Interactive Demo

A **first-principles physical model** that links tissue stiffness to a cell's nuclear form, YAP localization, and mechanogenomic state.

This notebook lets you **explore the virtual cell interactively**: choose a phenotype and set the physical parameters (substrate stiffness, time), and the model predicts the cell's state and renders it — as a fluorescence-style image and as a labeled cross-section.

> **How to use:** Run the cells top to bottom (Runtime → Run all). The code is hidden; you only interact with the controls. First run takes ~30 s to install.

Repository: https://github.com/Danpc11/mechanogenomic-virtual-cell

In [ ]:
#@title ⚙️ Setup — install & clone (run once, ~30 s) { display-mode: "form" }
import os, sys, subprocess

REPO = "https://github.com/Danpc11/mechanogenomic-virtual-cell.git"
if not os.path.exists("mechanogenomic-virtual-cell"):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "-q", REPO], check=True)

# install lightweight deps used by the demo
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "scipy", "matplotlib", "ipywidgets", "cairosvg"],
               check=False)

# put src/ and visualization/ on the path
BASE = os.path.abspath("mechanogenomic-virtual-cell")
for sub in ("src", "visualization"):
    p = os.path.join(BASE, sub)
    if p not in sys.path:
        sys.path.insert(0, p)

print("✅ Setup complete.")

In [ ]:
#@title ⚙️ Load the virtual cell & renderers (hidden) { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
import base64
import ipywidgets as widgets
from IPython.display import display, clear_output, SVG, HTML

from virtual_cell import VirtualCell
from mvirtual_cell import PHENOTYPES
import gene_module as gm
import render_cross_section_hq as rx
import cairosvg

# fluorescence renderer is optional — degrade gracefully if not present
try:
    import render_fluorescence as rf
    HAS_FLUO = True
except Exception:
    HAS_FLUO = False

PHENO_KEYS = [k for k in PHENOTYPES if k != "hepatocyte_posterior"]

def _svg_to_png_bytes(svg_str, width=760):
    return cairosvg.svg2png(bytestring=svg_str.encode(), output_width=width)

print("✅ Virtual cell loaded. Available phenotypes:", ", ".join(PHENO_KEYS))
if not HAS_FLUO:
    print("ℹ️ Fluorescence renderer not found in repo — using cross-section view only.")

## 🎛️ Explore the virtual cell

Use the controls below. The model recomputes the full cell state and its visualizations whenever you change a parameter.

In [ ]:
#@title 🔬 Interactive virtual cell (run, then use the controls) { display-mode: "form" }

phenotype = widgets.Dropdown(options=PHENO_KEYS, value="hepatocyte",
                             description="Phenotype:", style={"description_width": "initial"})
stiffness = widgets.FloatSlider(value=23.0, min=0.5, max=40.0, step=0.5,
                                description="Stiffness E (kPa):", continuous_update=False,
                                style={"description_width": "initial"}, layout=widgets.Layout(width="480px"))
time_h = widgets.FloatSlider(value=120.0, min=2.0, max=120.0, step=2.0,
                             description="Time t (h):", continuous_update=False,
                             style={"description_width": "initial"}, layout=widgets.Layout(width="480px"))
view = widgets.ToggleButtons(options=["Fluorescence", "Cross-section", "Both"],
                             value="Both", description="View:")

out = widgets.Output()

def _readout_html(s):
    return f"""
    <table style='font-family:monospace;border-collapse:collapse;margin-top:8px'>
      <tr><td style='padding:2px 14px 2px 0'>fibrosis stage</td><td><b>{s.fibrosis_stage}</b></td>
          <td style='padding:2px 14px 2px 24px'>nuclear area</td><td><b>{s.nuclear_area:.0f} µm²</b></td></tr>
      <tr><td>nuclear drive σ</td><td><b>{s.nuclear_drive:.1f}</b></td>
          <td style='padding-left:24px'>YAP N/C</td><td><b>{s.yap_nc:.2f}</b></td></tr>
      <tr><td>relaxation τ</td><td><b>{s.tau_h:.0f} h</b></td>
          <td style='padding-left:24px'>function index</td><td><b>{s.function_index:.2f}</b></td></tr>
    </table>"""

def update(*_):
    with out:
        clear_output(wait=True)
        cell = VirtualCell(phenotype.value)
        s = cell.simulate(stiffness.value, t=time_h.value)
        display(HTML(_readout_html(s)))

        v = view.value
        if v in ("Fluorescence", "Both") and HAS_FLUO:
            fig, ax = plt.subplots(1, 2, figsize=(8, 4), facecolor="black")
            ax[0].imshow(rf.render_if(s, mode="actin")); ax[0].axis("off")
            ax[0].set_title("F-actin", color="white", fontsize=11)
            ax[1].imshow(rf.render_if(s, mode="merge")); ax[1].axis("off")
            ax[1].set_title("actin / YAP / DAPI", color="white", fontsize=11)
            plt.tight_layout(); plt.show()

        if v in ("Cross-section", "Both") or not HAS_FLUO:
            svg = rx.render_svg(s, title=f"E = {stiffness.value:.0f} kPa · {s.fibrosis_stage}", uid="D")
            png = _svg_to_png_bytes(svg, width=680)
            b64 = base64.b64encode(png).decode()
            display(HTML(f'<img src="data:image/png;base64,{b64}" width="560"/>'))

        # top mechanosensitive genes
        top = sorted(s.gene_scores.items(), key=lambda kv: kv[1], reverse=True)[:6]
        bars = "".join(
            f"<div style='display:flex;align-items:center;gap:6px;font-family:monospace;font-size:12px'>"
            f"<span style='width:130px'>{g}</span>"
            f"<span style='display:inline-block;height:9px;width:{int(v*160)}px;background:#c0392b;border-radius:3px'></span>"
            f"<span>{v:.2f}</span></div>"
            for g, v in top)
        display(HTML(f"<b style='font-family:sans-serif'>Predicted gene activation</b>{bars}"))

for w in (phenotype, stiffness, time_h, view):
    w.observe(update, names="value")

controls = widgets.VBox([phenotype, stiffness, time_h, view])
display(controls, out)
update()

## 📈 Stiffness sweep

See how the whole mechanogenomic trajectory changes across the fibrosis range (F0 → F4) for the chosen phenotype.

In [ ]:
#@title 📈 Trajectory across fibrosis stages { display-mode: "form" }
pheno_sweep = "hepatocyte" #@param ["hepatocyte", "A549", "NHLF", "AT2_lung", "MCF10A", "MDA", "fibroblast"]
time_point = 120 #@param {type:"slider", min:2, max:120, step:2}

cell = VirtualCell(pheno_sweep)
Es = np.linspace(1, 30, 30)
states = [cell.simulate(E, t=time_point) for E in Es]
drive = [s.nuclear_drive for s in states]
yap = [s.yap_nc for s in states]
area = [s.nuclear_area for s in states]
func = [s.function_index for s in states]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, y, lab, col in zip(axes, [drive, yap, func],
                           ["nuclear drive σ", "YAP N/C", "function index"],
                           ["#6366F1", "#10B981", "#EF4444"]):
    ax.plot(Es, y, color=col, lw=2.5)
    ax.set_xlabel("stiffness E (kPa)"); ax.set_ylabel(lab)
    for cut, name in [(1,"F0"),(7,"F1"),(9.5,"F2"),(13,"F3"),(26,"F4")]:
        ax.axvline(cut, ls=":", color="#ccc", lw=1)
fig.suptitle(f"{pheno_sweep} — mechanogenomic trajectory at t = {time_point} h", fontsize=12)
fig.tight_layout(); plt.show()

---
### Notes

- **Hepatocyte** is the fully-calibrated reference case study (hepatic fibrosis); the other phenotypes are literature-anchored starting points.
- All visualizations are **generated from the model's own state outputs** — nucleus size from area, YAP localization from the N/C ratio, F-actin from the nuclear drive, etc. — not hand-drawn.
- This is a **hypothesis-generating** research tool, not a diagnostic.

If you use it, please cite the repository (see `CITATION.cff`).